In [ ]:
from compare import RESULTS_FILE_NAME, INPUT_FILE_NAME, ALGORITHMS, LABELS, LABELS_COMPRESSIONS, FIG_DIR, KS, ALG_OLD, ALG_MATCHTIGS, load_all_inputs, load_all_results
from compare_penalties import RESULTS_FILE_NAME as PENALTY_RESULTS_FILE_NAME

import matplotlib.pyplot as plt
import seaborn as sns
import math

LABELS_READABLE = ["Length of superstring", "Number of runs of ones", "Time [seconds]", "Memory [kB]", "Objective", "Relative objective"]

plt.rcParams["font.family"] = "serif"

In [ ]:
results = load_all_results("../" + RESULTS_FILE_NAME)
print(list(results)[0])
print(len(results))

penalty_results = load_all_results("../" + PENALTY_RESULTS_FILE_NAME)
print(list(penalty_results)[0])
print(len(penalty_results))

inputs = list(map(lambda x: tuple(x.split()), load_all_inputs("../" + INPUT_FILE_NAME)))
print(*inputs, sep="\n")

In [ ]:
LABELS_SHORT = LABELS[:]
LABELS_SHORT.pop(4)
LABELS_READABLE = ["Length of superstring", "Number of runs of ones", "Time [seconds]", "Memory [kB]", "Relative objective", "Relative compressed size"]

def print_stats_for_input(inp, save=False):
    run_penalty = int(inp[2])
    name = inp[3].replace("(p)", "pangenome").replace("_", " ").strip()
    if name == "-": return
    max_ks = list(filter(lambda x: x < int(inp[1]), KS))
    inp = inp[0]
    if "subsampled" in inp: return
    max_xs = []
    empty = True

    fig, axs = plt.subplots(2, 3)
    fig.set_figheight(9)
    fig.set_figwidth(13.5)
    fig.tight_layout(w_pad=2.5, h_pad=3.5)

    sns.lineplot(y=[1 for i in range(len(max_ks))], x=max_ks, ax=axs[-1, -1], marker="o", markeredgewidth=0)
    
    for alg in ALGORITHMS:
        rp = run_penalty if alg != "baseline" and alg != "global" else None
        for complements in (True, ):
            ys, xs = [list() for label in LABELS_SHORT], []
            cs = [list() for label in LABELS_COMPRESSIONS]
            for k in KS:
                if not (alg, inp, k, complements, rp) in results.keys(): continue
                empty = False

                d = results[(alg, inp, k, complements, rp)]
                xs.append(k)
                for i in range(len(LABELS_SHORT)):
                    ys[i].append(d[LABELS_SHORT[i]])
                for i in range(len(LABELS_COMPRESSIONS)):
                    if not (ALG_OLD, inp, k, complements, None) in results.keys():
                        cs[i].append(0)
                        continue
                    cs[i].append(d[LABELS_COMPRESSIONS[i]] / results[(ALG_OLD, inp, k, complements, None)][LABELS_COMPRESSIONS[i]]
                                 if alg != ALG_OLD else 1)

            if len(xs) > len(max_xs): max_xs = xs

            for i, label in enumerate(LABELS_SHORT):
                sns.lineplot(y=ys[i], x=xs, ax=axs[i // 3, i % 3],
                                marker="o" if complements else "s",
                                markeredgewidth=0, label=("GGMO" if alg == ALG_OLD else ("matchtigs" if alg == ALG_MATCHTIGS else "LOAC")))
            if alg != ALG_OLD:
                for i, label in enumerate(LABELS_COMPRESSIONS):
                    sns.lineplot(y=cs[i], x=xs, ax=axs[-1, -1],
                                    marker="o" if complements else "s",
                                    markeredgewidth=0, label=label)
    
    if empty:
        fig.clf()
        return
    
    # fig.suptitle(f"{name}, run penalty: {run_penalty}")
    print(f"{name}, run penalty: {run_penalty}")

    for i, label in enumerate(LABELS_READABLE):
        ax = axs[i // 3, i % 3]
        ax.set_title(label)
        ax.set(xticks=max_xs)
        ax.set_xlabel("k =")
        ax.xaxis.set_label_coords(-0.05, -0.025)
        if i in (1, 2, 3): ax.set_ylim(bottom=0)
    plt.legend()

    if save and not empty: fig.savefig(f"../{FIG_DIR}/{name}.pdf")


In [ ]:
for inp in inputs:
    print_stats_for_input(inp, save=True)